# J. Cole Discography Topic




# Introduction

J. Cole has been one of the most popular rappers in the 21st century with about 40 billion streams and 111.5 million sold units throughout his whole career. With his widespread popularity, he has been able to spread his ideas and thinking on many topics through his lyricism and songwriting. Many of his fans, including myself, find the way he is able to encapsulate his thoughts and criticisms of different topics to be intriguing and exciting. The unique way he is able to incorporate a large variety of themes in his discography while keeping the songs sonically interesting is what makes him distinguishable from other rappers. He is also known for his straightforward lyricism and storytelling where he is able to effectively paint a story in the listeners mind while also being able to convey deep introspective themes and messages to his audience. Throughout his career, he has produced and written hundreds of songs that have touched on topics such as racism, self doubt, love, and poverty. The question is: Can machine learning be used to understand and pick up these themes? If so, how specific can it get? What is the range of themes it is able to pick up? To be able to answer these questions, we are going to need to use natural language processing and machine learning. Natural language processing is a way in which machines are able to extract insights from corpora’s of text in human languages like English. While machine learning is a general branch of artificial intelligence that involves computers learning to detect patterns in datasets in order to make predictions on other data. On this project, I will be using *Blueprints for Text Analytics* by Jens Albrecht, Sidharth Ramachandran, and Christrian Winkler for inspiration of NLP techniques to use while processing the corpora of text at hand. With the use of natural language processing and machine learning, I am going to attempt to extract the themes and topics from the corpora of the song lyrics of J. Cole’s discography.


# Concerns

Before starting, there are some concerns that may hinder the results of the project. One of the concerns is that rappers, like J. Cole, often write their lyrics in metaphorical language. There are often double entendres and hidden meanings behind the lyrics. We are eventually going to tokenize and embed these lyrics into semantic vectors which are essentially a list of numbers that represent the meaning of the words in the lyrics. We have to keep this in mind as we choose the embedding model, as many embedding models use static embedding which assigns a word to just one meaning. There are other embedding models which use models that can detect double meanings like contextualized embedding models. Another concern that I have is that rap songs are not structured like regular text. Instead of paragraphs, there are verses and choruses that are often repeated and are way shorter than regular paragraphs. I am concerned that this different structure will hinder the model’s ability to extract the themes like the other corpora it has been trained on. Lastly, I am concerned that the size of the dataset might not be big enough to extract understandable themes. Although 230 songs is a lot, I know machine learning requires large datasets in the thousands and millions to be able to actually perform.

In [ ]:
!pip install textacy nltk bertopic umap-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import nltk
import json

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# Dataset and EDA

[j_cole.csv](https://drive.google.com/file/d/1dLcgZ1XfRiJGoysZaxB1oVboJ9Uh6hEq/view?usp=drive_link)

This dataset was created by scraping J. Cole lyrics from the [Genius API](https://genius.com/api-clients) using the [LyricsGenius](https://github.com/johnwmillr/lyricsgenius) Python library. With the use of these tools, I created a script that extracted the title, album, language, and lyrics for each song and put it into a Pandas Dataframe. The problem with using Genius as a source for the lyrics is that the lyrics are open source. This means that the lyrics are not verified by the actual artist and could have words that are wrong or misinterpreted. This could lead to the machine learning model misinterpreting the lyrics and giving the wrong topic. I also ran into the problem of people creating songs under J. Cole’s name, while the songs are not actually his or in an entirely different language. These were usually the songs that did not have an album associated with it. That is why I dropped all of the rows that did not have an album. This is a tradeoff because this deletes all of the features and singles that J. Cole released. This means that we are going to train the model purely off of J. Cole’s albums. This leaves us with about 226 songs in the corpora.



In [ ]:
df = pd.read_csv('/content/j_cole.csv', index_col=0)
df.head()

,Title,Album,Language,Lyrics
0,No Role Modelz,2014 Forest Hills Drive (10 Year Anniversary E...,en,"First things first: rest in peace, Uncle Phil\..."
1,She Knows,Born Sinner (Deluxe Version),en,"She knows\nShe knows, ayy\nBad things happen t..."
2,Power Trip,Born Sinner (Deluxe Version),en,Got me up all night\nAll I'm singin' is love s...
3,Wet Dreamz,2014 Forest Hills Drive (10 Year Anniversary E...,en,Cole\nCole world\nYeah\nLet me take y'all back...
4,Love Yourz,2014 Forest Hills Drive (10 Year Anniversary E...,en,"Hm, love yours\nHm, love yours\nNo such thing\..."


In [ ]:
df = df[df['Language'] == 'en']

In [ ]:
df.isna().sum()

,0
Title,0
Album,77
Language,0
Lyrics,0


In [ ]:
df = df.dropna()

In [ ]:
len(df)

226

# Preprocessing and EDA

While preprocessing the text, I had to make many decisions on what I wanted to include inside the training dataset of the model. First, I wanted to make sure that the text was normalized in a way so that the model did not have any trouble tokenizing and vectorizing the text. I got rid of any weird symbols like hyphenated words, new lines, and quotation marks. I also had to make decisions on what words would be “semantically insignificant” or what words would hinder the model from extracting the themes of the songs. These words that are “semantically insignificant” are called stop words in natural language processing. I decided to use nltk’s list stop word while adding on my own list of stop words based on the frequency and TF-IDF scores of the words. This whole process of preprocessing the data is adding both my own and nltk’s bias to the corpus as we are using our own bias to determine what words are "semantically insignificant” and hard to read for the model. For example, I decided to make all profanity stop words as it was my belief that they do not add any meaning to the text. That was a choice that I made to the dataset, but what did they add meaning to? That meaning was lost because of my decision. This just reinforces the idea that all models and statistical processes are inherently biased. As there is always an individual that had to make decision in every step of the process


In [ ]:
import textacy.preprocessing.normalize as tprep
import re

def preprocess(text):
    pt = tprep.quotation_marks(text)
    pt = tprep.hyphenated_words(pt)
    pt = tprep.unicode(pt)
    pt = re.sub(r'\[.*?\]', '', pt, flags=re.DOTALL)
    pt = re.sub(r'\(.*?\)', '', pt)
    pt = re.sub(r'(\w+)-\s*-(\w+)', r'\1\2', pt)
    pt = pt.replace('\n', ' ')
    pt = tprep.whitespace(pt)
    pt = pt.lower()
    return pt

In [ ]:
df["Preprocessed_Lyrics"] = df['Lyrics'].map(preprocess)

In [ ]:
frequency = df['Preprocessed_Lyrics'].str.lower().str.split(expand=True).stack().value_counts()
frequency.head(50)

,count
i,5127
the,4913
you,3513
a,3101
to,2506
and,2453
my,2379
i'm,1659
that,1569
it,1492


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=200
)

X = tfidf.fit_transform(df['Preprocessed_Lyrics'])

scores = pd.DataFrame({
    'word': tfidf.get_feature_names_out(),
    'score': X.toarray().mean(axis=0)
}).sort_values('score', ascending=False)

scores.head(50)

,word,score
123,nigga,0.137819
98,like,0.119522
124,niggas,0.114818
196,yeah,0.105201
68,got,0.096076
148,shit,0.094922
0,ain,0.093408
93,know,0.091500
42,don,0.084368
90,just,0.071658


In [ ]:
additional_stopwords = ["like", "yeah", "uh", "just", "ayy", "hey", "shit", "know", "fuck", "bitch", "got", "get", "see", "cause", "never", "let", "six", "two", "la", "niggas", "nigga"]

In [ ]:
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

# nltk.download('stopwords')

nltk_stopwords = stopwords.words('english')
total_stopwords = additional_stopwords + nltk_stopwords

vectorizer = CountVectorizer(stop_words=total_stopwords)


# Machine Learning Method Discussion and Analysis

The type of machine learning models that I decided to use to extract the themes of J. Cole's songs are topic models. Essentially, topic models are able to analyze texts to determine what group of tokens are most frequently seen together. Through clustering algorithms, they are able to group documents that share these groups of tokens. From these groups of tokens, I can infer what theme that topic could represent. My specific model of choice is BerTopic which is a topic model that uses BERT (Bidirectional Encoded Representations from Transformers) as the foundation for creating topics. BERT is special because it is bidirectional meaning that it could look both forward and backwards from a specific word in order to gain more context on what to predict. (Devlin, Jacob, et al.) The reason why I decided to choose Bertopic out all the other topic models was because of how modular the process is. (Grootendorst, Maarten.) It allows me to pick both embedding and clustering models seamlessly. I especially needed to be able to pick the embedding model because I wanted to use the Roberta Twitter embedding model. This embedding model is trained on a dataset of twitter tweets which allows it to embed slang which is important when analyzing rap lyrics as rap uses a lot of slang. It also allows me to tweak the parameters of the UMAP clustering algorithm so that it could cluster topics more effectively. In the end this just shows how machine learning models in general will always display results that are swayed to the bias of its creator. The creator, like me, had to make decisions based on their own experience and life on what models and parameters to use. In the end, a human still has to interpret the results of the model output and that adds more bias as the interpretation of the reader is inherently biased. In this case, I had to interpret how to make the topics Bertopic extracted out of the corpora into a more coherent topic that more people can understand. Many people could have interpreted differently than I did, but in the end I was the one to interpret it.


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("cardiffnlp/twitter-roberta-base-2022-154m")
input_ids = embedding_model.tokenizer(df.loc[1, "Preprocessed_Lyrics"], return_tensors = 'pt')['input_ids'][0]
tokens = embedding_model.tokenizer.convert_ids_to_tokens(input_ids)

print(tokens)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (861 > 512). Running this sequence through the model will result in indexing errors


['<s>', 'she', 'Ġknows', 'Ġshe', 'Ġknows', ',', 'Ġa', 'yy', 'Ġbad', 'Ġthings', 'Ġhappen', 'Ġto', 'Ġthe', 'Ġpeople', 'Ġyou', 'Ġlove', 'Ġand', 'Ġyou', 'Ġfind', 'Ġyourself', 'Ġpraying', 'Ġup', 'Ġto', 'Ġheaven', 'Ġabove', 'Ġbut', 'Ġhonestly', ',', 'Ġi', 'Ġnever', 'Ġhad', 'Ġmuch', 'Ġsympathy', "Ġ'", 'cause', 'Ġthose', 'Ġbad', 'Ġthings', ',', 'Ġi', 'Ġalways', 'Ġsaw', 'Ġthem', 'Ġcoming', 'Ġfor', 'Ġme', 'Ġi', "'m", 'Ġgonna', 'Ġrun', ',', 'Ġrun', 'Ġaway', ',', 'Ġrun', 'Ġ,', 'Ġrun', 'Ġaway', ',', 'Ġrun', 'Ġaway', 'Ġrun', 'Ġaway', 'Ġand', 'Ġnever', 'Ġcome', 'Ġback', 'Ġrun', ',', 'Ġrun', 'Ġaway', ',', 'Ġrun', ',', 'Ġrun', 'Ġaway', 'Ġ,', 'Ġrun', 'Ġaway', 'Ġshow', "Ġ'", 'em', 'Ġthat', 'Ġyour', 'Ġcolor', 'Ġis', 'Ġblack', 'Ġdamned', 'Ġif', 'Ġi', 'Ġdo', ',', 'Ġdamned', 'Ġif', 'Ġi', 'Ġdon', "'t", 'Ġyou', 'Ġknow', 'Ġi', 'Ġgot', 'Ġa', 'Ġgirl', 'Ġback', 'Ġhome', 'Ġyou', 'Ġgot', 'Ġa', 'Ġman', ',', 'Ġwhat', 'Ġyou', 'Ġwant', '?', 'Ġwhat', 'Ġyou', 'Ġwant', '?', 'Ġwhat', 'Ġthese', 'Ġbit', 'ches', 'Ġwant', 'Ġfro

In [ ]:
from bertopic import BERTopic
from umap import UMAP

# 49328570

umap_model = UMAP(
    n_neighbors = 5,
    n_components = 5,
    min_dist = 0,
    metric = 'cosine',
    random_state = 1
    )

topic_model = BERTopic(
    embedding_model = embedding_model,
    vectorizer_model=vectorizer,
    nr_topics=20,
    umap_model = umap_model
)

topics, prob = topic_model.fit_transform(df["Preprocessed_Lyrics"])

topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,79,-1_man_one_love_wanna,"[man, one, love, wanna, time, gon, back, go, m...","[lotta shit happens, like, being in show busin..."
1,0,23,0_count_way_oh_back,"[count, way, oh, back, loves, want, time, love...","[yeah, uh, yeah you gotta lis— you gotta follo..."
2,1,14,1_girl_bitchez_night_back,"[girl, bitchez, night, back, love, one, go, li...","[yeah, warm up la-la-la-la, la-la-la-la, la-la..."
3,2,42,2_man_tryna_go_gotta,"[man, tryna, go, gotta, back, city, tell, baby...","[yeah yeah, yeah, yeah yup ayy yeah ayy ayy, i..."
4,3,27,3_high_money_made_man,"[high, money, made, man, say, gettin, gon, sti...","[yeah, uh this for all my 'ville niggas, man, ..."
5,4,15,4_love_man_presidents_new,"[love, man, presidents, new, represent, money,...",[yeah fayettenam uh these niggas is playin' ru...
6,5,26,5_back_go_break_one,"[back, go, break, one, na, god, want, think, f...",[this for everybody out there that ever fell o...


# Graph and Results

The model resulted in extracting 5 themes out of the corpus of J. Cole’ discography. Before the final results, it gave 6 topics and a lot of outliers, so I had to tell the model to try to reduce the amount of outliers. After interpreting the results, with models with reduced outliers, I decided to merge 2 of the clusters because they were very similar. This resulted in the model that I found was most interpretable. I graphed plots of these clustered points at each step with final graph having the coherent names of the topics.


In [ ]:
for i in topic_model.get_topic_info()['Representation']:
    print(i)

['man', 'one', 'love', 'wanna', 'time', 'gon', 'back', 'go', 'make', 'still']
['count', 'way', 'oh', 'back', 'loves', 'want', 'time', 'love', 'come', 'thing']
['girl', 'bitchez', 'night', 'back', 'love', 'one', 'go', 'little', 'want', 'sky']
['man', 'tryna', 'go', 'gotta', 'back', 'city', 'tell', 'baby', 'girl', 'hit']
['high', 'money', 'made', 'man', 'say', 'gettin', 'gon', 'still', 'em', 'make']
['love', 'man', 'presidents', 'new', 'represent', 'money', 'york', 'last', 'say', 'smile']
['back', 'go', 'break', 'one', 'na', 'god', 'want', 'think', 'fly', 'survive']


In [ ]:
trained_embeddings = topic_model._extract_embeddings(df['Preprocessed_Lyrics'].to_numpy(), method="document")

topic_model.visualize_documents(docs=df['Title'].to_numpy(), embeddings=trained_embeddings, custom_labels=False)

In [ ]:
new_topics = topic_model.reduce_outliers(df['Preprocessed_Lyrics'].to_numpy(), topics, strategy = "embeddings")
topic_model.update_topics(df['Preprocessed_Lyrics'].to_numpy(), topics=new_topics, vectorizer_model=vectorizer)
topic_model.get_topic_info()

2026-06-11 19:48:16,621 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,0,23,0_count_way_oh_back,"[count, way, oh, back, time, want, loves, love...","[yeah, uh, yeah you gotta lis— you gotta follo..."
1,1,14,1_girl_bitchez_night_back,"[girl, bitchez, night, back, love, one, go, li...","[yeah, warm up la-la-la-la, la-la-la-la, la-la..."
2,2,42,2_man_tryna_go_gotta,"[man, tryna, go, gotta, back, city, tell, baby...","[yeah yeah, yeah, yeah yup ayy yeah ayy ayy, i..."
3,3,27,3_high_money_man_made,"[high, money, man, made, say, gon, still, gett...","[yeah, uh this for all my 'ville niggas, man, ..."
4,4,94,4_man_love_one_wanna,"[man, love, one, wanna, time, gon, back, make,...",[yeah fayettenam uh these niggas is playin' ru...
5,5,26,5_back_go_break_one,"[back, go, break, one, na, god, want, think, f...",[this for everybody out there that ever fell o...


In [ ]:
topic_model.visualize_documents(docs=df['Title'].to_numpy(), embeddings=trained_embeddings, custom_labels=False, title="Themes of J. Cole's Discography")

In [ ]:
df['topic'] = new_topics

df[['topic', 'Title']].sort_values('topic')

,topic,Title
1,0,She Knows
6,0,MIDDLE CHILD
5,0,Crooked Smile
4,0,Love Yourz
10,0,Kevin’s Heart
...,...,...
163,5,Lonely at the Top
147,5,The Badness
148,5,The Villest
198,5,Killers


In [ ]:
topic_model.merge_topics(df['Preprocessed_Lyrics'].to_numpy(), [0,1])

coherent_topic_labels = {
    0: "Love and Pride",
    1: "Ambition and Storytelling",
    2: "Legacy and Success",
    3: "Emotion and Intropection",
    4: "Faith"
}

topic_model.set_topic_labels(coherent_topic_labels)

In [ ]:
topic_model.visualize_documents(docs=df['Title'].to_numpy(), embeddings=trained_embeddings, custom_labels=True, title="Themes of J. Cole's Discography")

# Findings and Limitations

The resulting model found that the main themes J. Cole’s discography was Love and Pride, Ambition and Storytelling, Legacy and Success, Emotion and Introspection, and Faith. However, these findings do not feel like they were extracted by the machine learning model. It feels like I was the one who decided what the themes were. It felt like it would have been easier to analyze J. Cole’s music myself instead of trying to tell the machine how to do it for me. In the end, I was making all the decisions anyway. From curating the dataset to interpreting the results, it felt like I was just holding the hands of the machine find the theme’s I already thought J. Cole’s discography had. The limitations of the machine learning model is that it doesn’t learn as efficiently as it’s been hyped up to be. It takes minutes to run every time and you have to run it many times until you find the right parameters that you think give the best results. This project has opened my eyes the most to be able to see through AI Hype. From what I have learned from this project, AI is an attempt to automate the intelligence and skills that we already have, like being able to understand music.


# Works Cited

Albrecht, Jens, et al. Blueprints for Text Analytics Using Python: Machine Learning-Based Solutions for Common Real-World (NLP) Applications. 1st ed., O'Reilly Media, 2021


Grootendorst, Maarten. “BERTopic: Neural Topic Modeling with a Class-Based TF-IDF Procedure.” arXiv:2203.05794, arXiv, 11 Mar. 2022. arXiv.org, https://doi.org/10.48550/arXiv.2203.05794.


Devlin, Jacob, et al. “BERT: Pre-Training of Deep Bidirectional Transformers for Language Understanding.” arXiv:1810.04805, arXiv, 24 May 2019. arXiv.org, https://doi.org/10.48550/arXiv.1810.04805.





